In [3]:
import os
import numpy as np
import pandas as pd

# Geospatial libraries check karo
try:
    import rasterio
    print("rasterio:", rasterio.__version__)
except ImportError:
    print("rasterio NOT installed")

try:
    import whitebox
    print("whitebox: installed")
except ImportError:
    print("whitebox NOT installed")

try:
    import geopandas as gpd
    print("geopandas:", gpd.__version__)
except ImportError:
    print("geopandas NOT installed")

try:
    import imdlib as imd
    print("imdlib: installed")
except ImportError:
    print("imdlib NOT installed")

print("\nEnvironment check done!")

rasterio: 1.4.4
whitebox: installed
geopandas: 1.1.4
imdlib: installed

Environment check done!


In [2]:
!pip install rasterio whitebox geopandas imdlib


     ---------------------------------------- 0.0/25.7 MB ? eta -:--:--
     ---------------------------------------- 0.0/25.7 MB 1.9 MB/s eta 0:00:14
     ---------------------------------------- 0.1/25.7 MB 2.0 MB/s eta 0:00:14
     ---------------------------------------- 0.3/25.7 MB 2.5 MB/s eta 0:00:11
      --------------------------------------- 0.5/25.7 MB 2.8 MB/s eta 0:00:09
      --------------------------------------- 0.6/25.7 MB 3.1 MB/s eta 0:00:09
     - -------------------------------------- 0.7/25.7 MB 2.8 MB/s eta 0:00:09
     - -------------------------------------- 0.9/25.7 MB 2.9 MB/s eta 0:00:09
     - -------------------------------------- 1.0/25.7 MB 3.2 MB/s eta 0:00:08
     - -------------------------------------- 1.0/25.7 MB 3.2 MB/s eta 0:00:08
     - -------------------------------------- 1.0/25.7 MB 3.2 MB/s eta 0:00:08
     - -------------------------------------- 1.0/25.7 MB 3.2 MB/s eta 0:00:08
     - -------------------------------------- 1.0/25.7 MB 3


[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import zipfile
import os

zip_path = r"V:\C1_DEM_16B_2005-2014_v3_R-1_78E29N_h44m assam1.zip"
extract_dir = r"V:\flash_flood_ml\dem_data"

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_dir)

for root, dirs, files in os.walk(extract_dir):
    for f in files:
        print(os.path.join(root, f))

In [5]:
import zipfile
import os

zip_path = r"V:\C1_DEM_16B_2005-2014_v3_R-1_78E29N_h44m assam1.zip"
extract_dir = r"V:\flash_flood_ml\dem_data"

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_dir)

# Kya extract hua dekho
for root, dirs, files in os.walk(extract_dir):
    for f in files:
        print(os.path.join(root, f))

V:\flash_flood_ml\dem_data\cdnh44m_v3r1\cdnh44m.dbf
V:\flash_flood_ml\dem_data\cdnh44m_v3r1\cdnh44m.prj
V:\flash_flood_ml\dem_data\cdnh44m_v3r1\cdnh44m.shp
V:\flash_flood_ml\dem_data\cdnh44m_v3r1\cdnh44m.shp.xml
V:\flash_flood_ml\dem_data\cdnh44m_v3r1\cdnh44m.shx
V:\flash_flood_ml\dem_data\cdnh44m_v3r1\cdnh44m.tif
V:\flash_flood_ml\dem_data\cdnh44m_v3r1\cdnh44m.xml
V:\flash_flood_ml\dem_data\cdnh44m_v3r1\policy.txt
V:\flash_flood_ml\dem_data\cdnh44m_v3r1\readme.txt


In [6]:
import rasterio
import numpy as np

dem_path = r"V:\flash_flood_ml\dem_data\cdnh44m_v3r1\cdnh44m.tif"

with rasterio.open(dem_path) as ds:
    print("Width x Height:", ds.width, "x", ds.height)
    print("CRS:", ds.crs)
    print("Bounds:", ds.bounds)
    print("Resolution:", ds.res)
    print("Dtype:", ds.dtypes)
    
    elevation = ds.read(1)
    print("\nElevation stats:")
    print("Min:", elevation.min(), "m")
    print("Max:", elevation.max(), "m")
    print("Mean:", round(elevation.mean(), 2), "m")

Width x Height: 3600 x 3600
CRS: EPSG:4326
Bounds: BoundingBox(left=77.99986111111112, bottom=29.000138888888888, right=78.99986111111112, top=30.000138888888888)
Resolution: (0.0002777777777777778, 0.0002777777777777778)
Dtype: ('int16',)

Elevation stats:
Min: 40 m
Max: 2633 m
Mean: 393.73 m


In [7]:
import whitebox

wbt = whitebox.WhiteboxTools()
wbt.set_working_dir(r"V:\flash_flood_ml\dem_data\cdnh44m_v3r1")
wbt.verbose = False

# Step 1: Depressions fill karo (sinks/pits ko hataana, warna flow-routing todta hai)
wbt.fill_depressions("cdnh44m.tif", "dem_filled.tif")
print("✓ Depressions filled")

# Step 2: Flow-direction (D8 algorithm)
wbt.d8_pointer("dem_filled.tif", "flow_dir.tif")
print("✓ Flow direction computed")

# Step 3: Flow-accumulation
wbt.d8_flow_accumulation("dem_filled.tif", "flow_accum.tif")
print("✓ Flow accumulation computed")

# Step 4: Slope
wbt.slope("dem_filled.tif", "slope.tif")
print("✓ Slope computed")

print("\nAll terrain-derivative rasters generated!")

<urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1007)>
Trying backup link ...
Decompressing WhiteboxTools_win_amd64.zip ...
WhiteboxTools package directory: V:\flash_flood_ml\venv\lib\site-packages\whitebox
✓ Depressions filled
✓ Flow direction computed
✓ Flow accumulation computed
✓ Slope computed

All terrain-derivative rasters generated!


In [8]:
import rasterio
import numpy as np

# Slope check karo
with rasterio.open(r"V:\flash_flood_ml\dem_data\cdnh44m_v3r1\slope.tif") as ds:
    slope_data = ds.read(1)
    valid_slope = slope_data[slope_data >= 0]  # negative/nodata hatao
    print("Slope (degrees):")
    print("  Min:", round(valid_slope.min(), 2))
    print("  Max:", round(valid_slope.max(), 2))
    print("  Mean:", round(valid_slope.mean(), 2))

# Flow-accumulation check karo
with rasterio.open(r"V:\flash_flood_ml\dem_data\cdnh44m_v3r1\flow_accum.tif") as ds:
    flow_data = ds.read(1)
    valid_flow = flow_data[flow_data >= 0]
    print("\nFlow Accumulation:")
    print("  Min:", valid_flow.min())
    print("  Max:", valid_flow.max())
    print("  Mean:", round(valid_flow.mean(), 2))

Slope (degrees):
  Min: 0.0
  Max: 89.99
  Mean: 24.45

Flow Accumulation:
  Min: 1.0
  Max: 5311.0
  Mean: 5.68


In [18]:
# Step A: Stream network nikalo (high flow-accumulation wale cells = streams)
wbt.extract_streams(
    "flow_accum.tif",
    "streams.tif",
    threshold=1200  # tune karna padega — flow_accum threshold jisse "stream" define ho
)
print("✓ Stream network extracted")

# Step B: Streams ko unique-IDs do (stream-link identification)
wbt.stream_link_identifier(
    "flow_dir.tif",
    "streams.tif",
    "stream_links.tif"
)
print("✓ Stream links identified")

# Step C: In stream-links se sub-catchments/watersheds banao
wbt.watershed(
    "flow_dir.tif",
    "stream_links.tif",
    "catchments.tif"
)
print("✓ Catchments delineated")

✓ Stream network extracted
✓ Stream links identified
✓ Catchments delineated


In [14]:
import rasterio
import numpy as np

with rasterio.open(r"V:\flash_flood_ml\dem_data\cdnh44m_v3r1\catchments.tif") as ds:
    catchment_data = ds.read(1)
    unique_catchments = np.unique(catchment_data)
    # 0 ya negative values usually "no data" hote hain
    valid_catchments = unique_catchments[unique_catchments > 0]
    
    print("Total unique catchments:", len(valid_catchments))
    print("Catchment IDs (first 20):", valid_catchments[:20])
    
    # Har catchment ka size (pixel count) dekho
    sizes = []
    for cid in valid_catchments:
        size = (catchment_data == cid).sum()
        sizes.append(size)
    
    sizes = np.array(sizes)
    print("\nCatchment sizes (pixel count):")
    print("  Min:", sizes.min())
    print("  Max:", sizes.max())
    print("  Mean:", round(sizes.mean(), 1))
    print("  Median:", np.median(sizes))

Total unique catchments: 120
Catchment IDs (first 20): [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20]

Catchment sizes (pixel count):
  Min: 16
  Max: 3971
  Mean: 1656.0
  Median: 1532.0


In [20]:
import rasterio
import numpy as np

MIN_SIZE = 200

with rasterio.open(r"V:\flash_flood_ml\dem_data\cdnh44m_v3r1\catchments_v3.tif") as ds:
    catchment_data_v3 = ds.read(1)

unique_v3 = np.unique(catchment_data_v3)
valid_v3 = unique_v3[unique_v3 > 0]

sizes_dict = {cid: (catchment_data_v3 == cid).sum() for cid in valid_v3}
valid_catchments_filtered = [cid for cid, size in sizes_dict.items() if size >= MIN_SIZE]

print("Before filter:", len(valid_v3), "catchments")
print("After filter (>=200 pixels):", len(valid_catchments_filtered), "catchments")

filtered_sizes = [sizes_dict[cid] for cid in valid_catchments_filtered]
print("Min:", min(filtered_sizes), "Max:", max(filtered_sizes), "Mean:", round(np.mean(filtered_sizes),1))

Before filter: 120 catchments
After filter (>=200 pixels): 119 catchments
Min: 846 Max: 3971 Mean: 1669.7


In [17]:
import os

folder = r"V:\flash_flood_ml\dem_data\cdnh44m_v3r1"

for f in os.listdir(folder):

    print(f)
    

catchments.tif
cdnh44m.dbf
cdnh44m.prj
cdnh44m.shp
cdnh44m.shp.xml
cdnh44m.shx
cdnh44m.tif
cdnh44m.xml
dem_filled.tif
flow_accum.tif
flow_dir.tif
policy.txt
readme.txt
slope.tif
streams.tif
stream_links.tif


In [19]:
import whitebox

wbt = whitebox.WhiteboxTools()
wbt.set_working_dir(r"V:\flash_flood_ml\dem_data\cdnh44m_v3r1")
wbt.verbose = False

wbt.extract_streams("flow_accum.tif", "streams_v3.tif", threshold=1200)
wbt.stream_link_identifier("flow_dir.tif", "streams_v3.tif", "stream_links_v3.tif")
wbt.watershed("flow_dir.tif", "stream_links_v3.tif", "catchments_v3.tif")

print("Done — files created")

import os
folder = r"V:\flash_flood_ml\dem_data\cdnh44m_v3r1"
print("catchments_v3.tif" in os.listdir(folder))

Done — files created
True


In [21]:
import pandas as pd

# Slope aur flow-accum rasters load karo
with rasterio.open(r"V:\flash_flood_ml\dem_data\cdnh44m_v3r1\slope.tif") as ds:
    slope_data = ds.read(1)

with rasterio.open(r"V:\flash_flood_ml\dem_data\cdnh44m_v3r1\flow_accum.tif") as ds:
    flow_data = ds.read(1)

# Har catchment ke liye average nikalo
records = []
for cid in valid_catchments_filtered:
    mask = (catchment_data_v3 == cid)
    records.append({
        'catchment_id': f"C{int(cid)}",
        'pixel_count': mask.sum(),
        'area_km2': round(mask.sum() * (30*30) / 1e6, 3),  # ~30m resolution assume
        'mean_slope': round(slope_data[mask].mean(), 2),
        'mean_flow_accum': round(flow_data[mask].mean(), 2),
        'max_flow_accum': round(flow_data[mask].max(), 2),
    })

catchment_features = pd.DataFrame(records)
print("Total catchments in feature-table:", len(catchment_features))
catchment_features.head(10)

Total catchments in feature-table: 119


,catchment_id,pixel_count,area_km2,mean_slope,mean_flow_accum,max_flow_accum
0,C1,1722,1.550,67.019997,47.029999,1722.0
1,C2,1524,1.372,72.199997,31.790001,1524.0
2,C3,2046,1.841,73.620003,37.750000,2046.0
3,C4,1230,1.107,70.610001,39.680000,1230.0
4,C5,2450,2.205,71.529999,38.959999,2450.0
5,C6,1249,1.124,69.629997,29.389999,1249.0
6,C7,1345,1.210,71.599998,27.370001,1345.0
7,C8,1424,1.282,72.870003,36.980000,1424.0
8,C9,1634,1.471,70.849998,47.639999,1634.0
9,C10,1643,1.479,75.589996,29.690001,1643.0


In [22]:
with rasterio.open(r"V:\flash_flood_ml\dem_data\cdnh44m_v3r1\streams_v3.tif") as ds:
    stream_data = ds.read(1)
    pixel_size_km = ds.res[0] * 111  # degree-to-km approx (rough, EPSG:4326)

drainage_records = []
for cid in valid_catchments_filtered:
    mask = (catchment_data_v3 == cid)
    stream_pixels_in_catchment = ((stream_data > 0) & mask).sum()
    area_km2 = mask.sum() * (30*30) / 1e6
    stream_length_km = stream_pixels_in_catchment * (30/1000)  # rough estimate
    drainage_density = round(stream_length_km / area_km2, 3) if area_km2 > 0 else 0
    drainage_records.append({'catchment_id': f"C{int(cid)}", 'drainage_density': drainage_density})

drainage_df = pd.DataFrame(drainage_records)
catchment_features = catchment_features.merge(drainage_df, on='catchment_id')
catchment_features.head(10)

,catchment_id,pixel_count,area_km2,mean_slope,mean_flow_accum,max_flow_accum,drainage_density
0,C1,1722,1.550,67.019997,47.029999,1722.0,0.213
1,C2,1524,1.372,72.199997,31.790001,1524.0,0.219
2,C3,2046,1.841,73.620003,37.750000,2046.0,0.407
3,C4,1230,1.107,70.610001,39.680000,1230.0,0.027
4,C5,2450,2.205,71.529999,38.959999,2450.0,0.259
5,C6,1249,1.124,69.629997,29.389999,1249.0,0.080
6,C7,1345,1.210,71.599998,27.370001,1345.0,0.099
7,C8,1424,1.282,72.870003,36.980000,1424.0,0.375
8,C9,1634,1.471,70.849998,47.639999,1634.0,0.592
9,C10,1643,1.479,75.589996,29.690001,1643.0,0.203


In [26]:
import zipfile
import os

# Apne local paths yahan daalo (jahan zip files save hain, V:\ ke andar)
assam_zips = {
    "90E25N": r"V:\C1_DEM_16B_2005-2014_v3_R-1_90E25N_g46m_ASSAM_4.zip",
    "91E25N": r"V:\C1_DEM_16B_2005-2014_v3_R-1_91E25N_g46n_ASSAM_5.zip",
    "92E24N": r"V:\C1_DEM_16B_2005-2014_v3_R-1_92E24N_g46u_ASSAM_1.zip",
    "93E24N": r"V:\C1_DEM_16B_2005-2014_v3_R-1_93E24N_g46v_ASSAM_2.zip",
    "94E24N": r"V:\C1_DEM_16B_2005-2014_v3_R-1_94E24N_g46w_ASAM_3.zip",
}

extract_base = r"V:\flash_flood_ml\dem_data_assam"
os.makedirs(extract_base, exist_ok=True)

for tile_name, zip_path in assam_zips.items():
    out_dir = os.path.join(extract_base, tile_name)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(out_dir)
    print(f"✓ Extracted {tile_name}")

print("\nAll tiles extracted!")

FileNotFoundError: [Errno 2] No such file or directory: 'V:\\C1_DEM_16B_2005-2014_v3_R-1_90E25N_g46m_ASSAM_4.zip'

In [27]:
import os

folder_path = "V:\\"

print("Items in V:\\\n")

for f in os.listdir(folder_path):
    full_path = os.path.join(folder_path, f)

    if os.path.isdir(full_path):
        print("📁 FOLDER :", f)
    else:
        print("📄 FILE   :", f)

Items in V:\

📁 FOLDER : $RECYCLE.BIN
📄 FILE   : Adarsh Baal Vidyalaya S01 (Ep.01-07) (2026) Hindi Completed Web Series HEVC 720p ESub.mkv
📄 FILE   : ani.mp4
📄 FILE   : animate1.mp4
📄 FILE   : animate2.mp4
📁 FOLDER : AnimatedWebsite
📄 FILE   : animation3.mp4
📄 FILE   : ASV.pptx
📄 FILE   : atul.pdf
📄 FILE   : C1_DEM_16B_2005-2014_v3_R-1_78E29N_h44m assam1.zip
📄 FILE   : C1_DEM_16B_2005-2014_v3_R-1_78E29N_h44m sih1.zip
📄 FILE   : C1_DEM_16B_2005-2014_v3_R-1_90E25N_g46m ASSAM 4.zip
📄 FILE   : C1_DEM_16B_2005-2014_v3_R-1_91E25N_g46n ASSAM 5.zip
📄 FILE   : C1_DEM_16B_2005-2014_v3_R-1_92E24N_g46u ASSAM 1.zip
📄 FILE   : C1_DEM_16B_2005-2014_v3_R-1_93E24N_g46v ASSAM 2.zip
📄 FILE   : C1_DEM_16B_2005-2014_v3_R-1_94E24N_g46w ASAM 3.zip
📁 FOLDER : contagiongrid
📁 FOLDER : contagiongrid 2
📄 FILE   : decode sih.pptx
📄 FILE   : decode sih_fiinal.pptx
📄 FILE   : decode.pdf
📄 FILE   : decodesih.pdf
📁 FOLDER : Downloads
📄 FILE   : Ek Din (2026) Hindi Movie HD ESub 720p HEVC.mkv
📁 FOLDER : figma
📁 FOLDER

In [28]:
import os
import zipfile

base_path = "V:\\"
extract_base = r"V:\flash_flood_ml\data\raw"

os.makedirs(extract_base, exist_ok=True)

# Assam/Asam wali ZIP files automatically find karo
assam_zips = [
    f for f in os.listdir(base_path)
    if f.lower().endswith(".zip")
    and ("assam" in f.lower() or "asam" in f.lower())
]

print("Datasets found:\n")

for i, f in enumerate(assam_zips, 1):
    print(f"{i}. {f}")

Datasets found:

1. C1_DEM_16B_2005-2014_v3_R-1_78E29N_h44m assam1.zip
2. C1_DEM_16B_2005-2014_v3_R-1_90E25N_g46m ASSAM 4.zip
3. C1_DEM_16B_2005-2014_v3_R-1_91E25N_g46n ASSAM 5.zip
4. C1_DEM_16B_2005-2014_v3_R-1_92E24N_g46u ASSAM 1.zip
5. C1_DEM_16B_2005-2014_v3_R-1_93E24N_g46v ASSAM 2.zip
6. C1_DEM_16B_2005-2014_v3_R-1_94E24N_g46w ASAM 3.zip


In [1]:
extracted_tiles = {}

for f in assam_zips:
    zip_path = os.path.join(base_path, f)
    tile_name = f.split("_R-1_")[1].split("_")[0]  # e.g. "90E25N"
    out_dir = os.path.join(extract_base, tile_name)
    
    if tile_name in extracted_tiles:
        print(f"Skipping duplicate tile: {tile_name}")
        continue
    
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(out_dir)
    
    extracted_tiles[tile_name] = out_dir
    print(f"✓ Extracted {tile_name} → {out_dir}")

print(f"\nTotal unique tiles extracted: {len(extracted_tiles)}")

NameError: name 'assam_zips' is not defined

In [2]:
import os
import zipfile

base_path = "V:\\"
extract_base = r"V:\flash_flood_ml\data\raw"
os.makedirs(extract_base, exist_ok=True)

assam_zips = [
    f for f in os.listdir(base_path)
    if f.lower().endswith(".zip")
    and ("assam" in f.lower() or "asam" in f.lower())
]

print("Found zips:", len(assam_zips))

extracted_tiles = {}

for f in assam_zips:
    zip_path = os.path.join(base_path, f)
    tile_name = f.split("_R-1_")[1].split("_")[0]  # e.g. "90E25N"
    out_dir = os.path.join(extract_base, tile_name)
    
    if tile_name in extracted_tiles:
        print(f"Skipping duplicate tile: {tile_name}")
        continue
    
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(out_dir)
    
    extracted_tiles[tile_name] = out_dir
    print(f"✓ Extracted {tile_name} → {out_dir}")

print(f"\nTotal unique tiles extracted: {len(extracted_tiles)}")

Found zips: 6
✓ Extracted 78E29N → V:\flash_flood_ml\data\raw\78E29N
✓ Extracted 90E25N → V:\flash_flood_ml\data\raw\90E25N
✓ Extracted 91E25N → V:\flash_flood_ml\data\raw\91E25N
✓ Extracted 92E24N → V:\flash_flood_ml\data\raw\92E24N
✓ Extracted 93E24N → V:\flash_flood_ml\data\raw\93E24N
✓ Extracted 94E24N → V:\flash_flood_ml\data\raw\94E24N

Total unique tiles extracted: 6


In [3]:
tile_tifs = {}

for tile_name, folder in extracted_tiles.items():
    for root, dirs, files in os.walk(folder):
        for f in files:
            if f.lower().endswith(".tif") and "xml" not in f.lower():
                tile_tifs[tile_name] = os.path.join(root, f)
                break

print("TIF files found:\n")
for tile, path in tile_tifs.items():
    print(f"{tile}: {path}")

TIF files found:

78E29N: V:\flash_flood_ml\data\raw\78E29N\cdnh44m_v3r1\cdnh44m.tif
90E25N: V:\flash_flood_ml\data\raw\90E25N\cdng46m_v3r1\cdng46m.tif
91E25N: V:\flash_flood_ml\data\raw\91E25N\cdng46n_v3r1\cdng46n.tif
92E24N: V:\flash_flood_ml\data\raw\92E24N\cdng46u_v3r1\cdng46u.tif
93E24N: V:\flash_flood_ml\data\raw\93E24N\cdng46v_v3r1\cdng46v.tif
94E24N: V:\flash_flood_ml\data\raw\94E24N\cdng46w_v3r1\cdng46w.tif


In [4]:
from rasterio.merge import merge
import rasterio

assam_tile_names = ["90E25N", "91E25N", "92E24N", "93E24N", "94E24N"]
assam_tif_paths = [tile_tifs[t] for t in assam_tile_names]

# Sab tiles ko open karo
src_files = [rasterio.open(p) for p in assam_tif_paths]

# Merge karo
mosaic, out_transform = merge(src_files)

# Metadata copy karke update karo
out_meta = src_files[0].meta.copy()
out_meta.update({
    "driver": "GTiff",
    "height": mosaic.shape[1],
    "width": mosaic.shape[2],
    "transform": out_transform
})

# Save karo
assam_dem_path = r"V:\flash_flood_ml\data\raw\assam_merged_dem.tif"
with rasterio.open(assam_dem_path, "w", **out_meta) as dest:
    dest.write(mosaic)

# Files close karo
for src in src_files:
    src.close()

print("✓ Assam DEM mosaic saved:", assam_dem_path)
print("Merged shape:", mosaic.shape)

✓ Assam DEM mosaic saved: V:\flash_flood_ml\data\raw\assam_merged_dem.tif
Merged shape: (1, 7200, 18000)


In [5]:
with rasterio.open(assam_dem_path) as ds:
    print("Width x Height:", ds.width, "x", ds.height)
    print("CRS:", ds.crs)
    print("Bounds:", ds.bounds)
    
    elevation = ds.read(1)
    valid_elev = elevation[elevation > -1000]  # nodata/garbage values hatao agar hon
    print("\nElevation stats:")
    print("Min:", valid_elev.min(), "m")
    print("Max:", valid_elev.max(), "m")
    print("Mean:", round(valid_elev.mean(), 2), "m")

Width x Height: 18000 x 7200
CRS: EPSG:4326
Bounds: BoundingBox(left=89.99986111111112, bottom=24.000138888888888, right=94.99986111111112, top=26.000138888888888)

Elevation stats:
Min: -113 m
Max: 2285 m
Mean: 210.3 m


In [7]:
import whitebox

wbt = whitebox.WhiteboxTools()
wbt.set_working_dir(r"V:\flash_flood_ml\data\raw")
wbt.verbose = False

wbt.fill_depressions("assam_merged_dem.tif", "assam_dem_filled.tif")
print("✓ Depressions filled")

wbt.d8_pointer("assam_dem_filled.tif", "assam_flow_dir.tif")
print("✓ Flow direction computed")

wbt.d8_flow_accumulation("assam_dem_filled.tif", "assam_flow_accum.tif")
print("✓ Flow accumulation computed")

wbt.slope("assam_dem_filled.tif", "assam_slope.tif")
print("✓ Slope computed")

print("\nAll Assam terrain-derivative rasters generated!")

✓ Depressions filled
✓ Flow direction computed
✓ Flow accumulation computed
✓ Slope computed

All Assam terrain-derivative rasters generated!


In [8]:
with rasterio.open(r"V:\flash_flood_ml\data\raw\assam_slope.tif") as ds:
    slope_data = ds.read(1)
    valid_slope = slope_data[slope_data >= 0]
    print("Assam Slope (degrees):")
    print("  Min:", round(valid_slope.min(), 2))
    print("  Max:", round(valid_slope.max(), 2))
    print("  Mean:", round(valid_slope.mean(), 2))

with rasterio.open(r"V:\flash_flood_ml\data\raw\assam_flow_accum.tif") as ds:
    flow_data = ds.read(1)
    valid_flow = flow_data[flow_data >= 0]
    print("\nAssam Flow Accumulation:")
    print("  Min:", valid_flow.min())
    print("  Max:", valid_flow.max())
    print("  Mean:", round(valid_flow.mean(), 2))

Assam Slope (degrees):
  Min: 0.0
  Max: 88.44
  Mean: 7.52

Assam Flow Accumulation:
  Min: 1.0
  Max: 86131320.0
  Mean: 50772.98


In [9]:
import numpy as np

percentiles = [50, 90, 95, 99, 99.5, 99.9]
for p in percentiles:
    val = np.percentile(valid_flow, p)
    print(f"{p}th percentile: {val:,.0f}")

50th percentile: 10
90th percentile: 2,320
95th percentile: 3,321
99th percentile: 251,945
99.5th percentile: 1,390,975
99.9th percentile: 11,476,906


In [10]:
wbt.extract_streams(
    "assam_flow_accum.tif",
    "assam_streams.tif",
    threshold=50000
)
print("✓ Streams extracted")

wbt.stream_link_identifier("assam_flow_dir.tif", "assam_streams.tif", "assam_stream_links.tif")
print("✓ Stream links identified")

wbt.watershed("assam_flow_dir.tif", "assam_stream_links.tif", "assam_catchments.tif")
print("✓ Catchments delineated")

✓ Streams extracted
✓ Stream links identified
✓ Catchments delineated


In [11]:
with rasterio.open(r"V:\flash_flood_ml\data\raw\assam_catchments.tif") as ds:
    assam_catchment_data = ds.read(1)

unique_assam = np.unique(assam_catchment_data)
valid_assam = unique_assam[unique_assam > 0]

print("Total unique catchments:", len(valid_assam))

sizes_assam = np.array([(assam_catchment_data == cid).sum() for cid in valid_assam[:200]])  # pehle 200 check karo, sab ka size-calc slow hoga
print("Sample sizes (first 200 catchments):")
print("  Min:", sizes_assam.min())
print("  Max:", sizes_assam.max())
print("  Mean:", round(sizes_assam.mean(), 1))

Total unique catchments: 973
Sample sizes (first 200 catchments):
  Min: 50233
  Max: 401444
  Mean: 109488.1


In [12]:
with rasterio.open(r"V:\flash_flood_ml\data\raw\assam_slope.tif") as ds:
    assam_slope_data = ds.read(1)

with rasterio.open(r"V:\flash_flood_ml\data\raw\assam_flow_accum.tif") as ds:
    assam_flow_data = ds.read(1)

with rasterio.open(r"V:\flash_flood_ml\data\raw\assam_streams.tif") as ds:
    assam_stream_data = ds.read(1)

# Pehle sabhi catchment-sizes calculate karo taaki filter kar sakein
print("Calculating sizes for all 973 catchments... (thoda time lagega)")
assam_sizes_dict = {cid: (assam_catchment_data == cid).sum() for cid in valid_assam}
print("Done!")

# Chhote noise-catchments filter karo (minimum size threshold)
MIN_SIZE_ASSAM = 5000  # ~4.5 km2, Assam ke pixel-scale ke hisaab se
valid_assam_filtered = [cid for cid, size in assam_sizes_dict.items() if size >= MIN_SIZE_ASSAM]
print(f"After filter: {len(valid_assam_filtered)} catchments (from {len(valid_assam)})")

Calculating sizes for all 973 catchments... (thoda time lagega)
Done!
After filter: 931 catchments (from 973)


In [14]:
import pandas as pd
records_assam = []

for cid in valid_assam_filtered:
    mask = (assam_catchment_data == cid)
    stream_pixels = ((assam_stream_data > 0) & mask).sum()
    area_km2 = mask.sum() * (30*30) / 1e6
    stream_length_km = stream_pixels * (30/1000)
    drainage_density = round(stream_length_km / area_km2, 3) if area_km2 > 0 else 0
    
    records_assam.append({
        'catchment_id': f"A{int(cid)}",  # "A" prefix taaki Uttarakhand ke "C" se distinct rahe
        'pixel_count': mask.sum(),
        'area_km2': round(area_km2, 3),
        'mean_slope': round(assam_slope_data[mask].mean(), 2),
        'mean_flow_accum': round(assam_flow_data[mask].mean(), 2),
        'max_flow_accum': round(assam_flow_data[mask].max(), 2),
        'drainage_density': drainage_density,
    })

assam_features = pd.DataFrame(records_assam)
print("Total Assam catchments in feature-table:", len(assam_features))
assam_features.head(10)

Total Assam catchments in feature-table: 931


,catchment_id,pixel_count,area_km2,mean_slope,mean_flow_accum,max_flow_accum,drainage_density
0,A1,53031,47.728,6.93,240.449997,53031.0,0.003
1,A2,71861,64.675,0.00,4664.359863,71861.0,1.667
2,A3,170284,153.256,0.00,5081.660156,170284.0,0.608
3,A4,78447,70.602,0.00,2070.709961,78447.0,0.001
4,A5,106955,96.260,0.00,1792.410034,106955.0,0.002
5,A6,107716,96.944,0.00,1805.099976,107716.0,0.002
6,A7,90809,81.728,8.04,262.309998,90809.0,0.008
7,A8,77669,69.902,0.00,4697.370117,77669.0,1.528
8,A9,63859,57.473,0.00,4251.649902,63859.0,1.755
9,A10,96887,87.198,0.00,1914.489990,96887.0,0.002


In [15]:
combined_terrain_features = pd.concat([catchment_features, assam_features], ignore_index=True)
combined_terrain_features['region'] = combined_terrain_features['catchment_id'].apply(
    lambda x: 'Uttarakhand' if x.startswith('C') else 'Assam'
)

print("Total combined catchments:", len(combined_terrain_features))
print(combined_terrain_features['region'].value_counts())
combined_terrain_features.to_csv(r"V:\flash_flood_ml\data\raw\terrain_features_combined.csv", index=False)
print("\n✓ Saved to terrain_features_combined.csv")

NameError: name 'catchment_features' is not defined

In [16]:
import os
print(os.listdir(r"V:\flash_flood_ml\dem_data\cdnh44m_v3r1"))

['catchments.tif', 'catchments_v3.tif', 'cdnh44m.dbf', 'cdnh44m.prj', 'cdnh44m.shp', 'cdnh44m.shp.xml', 'cdnh44m.shx', 'cdnh44m.tif', 'cdnh44m.xml', 'dem_filled.tif', 'flow_accum.tif', 'flow_dir.tif', 'policy.txt', 'readme.txt', 'slope.tif', 'streams.tif', 'streams_v3.tif', 'stream_links.tif', 'stream_links_v3.tif']


In [17]:
import rasterio
import numpy as np
import pandas as pd

# Uttarakhand rasters reload karo
with rasterio.open(r"V:\flash_flood_ml\dem_data\cdnh44m_v3r1\catchments_v3.tif") as ds:
    catchment_data_v3 = ds.read(1)

with rasterio.open(r"V:\flash_flood_ml\dem_data\cdnh44m_v3r1\slope.tif") as ds:
    slope_data = ds.read(1)

with rasterio.open(r"V:\flash_flood_ml\dem_data\cdnh44m_v3r1\flow_accum.tif") as ds:
    flow_data = ds.read(1)

with rasterio.open(r"V:\flash_flood_ml\dem_data\cdnh44m_v3r1\streams_v3.tif") as ds:
    stream_data = ds.read(1)

# Valid catchments nikalo aur filter karo (jaisa pehle kiya tha)
unique_v3 = np.unique(catchment_data_v3)
valid_v3 = unique_v3[unique_v3 > 0]

MIN_SIZE = 200
sizes_dict = {cid: (catchment_data_v3 == cid).sum() for cid in valid_v3}
valid_catchments_filtered = [cid for cid, size in sizes_dict.items() if size >= MIN_SIZE]

# Feature-table dobara banao
records = []
for cid in valid_catchments_filtered:
    mask = (catchment_data_v3 == cid)
    stream_pixels_in_catchment = ((stream_data > 0) & mask).sum()
    area_km2 = mask.sum() * (30*30) / 1e6
    stream_length_km = stream_pixels_in_catchment * (30/1000)
    drainage_density = round(stream_length_km / area_km2, 3) if area_km2 > 0 else 0
    
    records.append({
        'catchment_id': f"C{int(cid)}",
        'pixel_count': mask.sum(),
        'area_km2': round(area_km2, 3),
        'mean_slope': round(slope_data[mask].mean(), 2),
        'mean_flow_accum': round(flow_data[mask].mean(), 2),
        'max_flow_accum': round(flow_data[mask].max(), 2),
        'drainage_density': drainage_density,
    })

catchment_features = pd.DataFrame(records)
print("Uttarakhand catchments rebuilt:", len(catchment_features))

# Turant save kar do taaki dobara na khona pade
catchment_features.to_csv(r"V:\flash_flood_ml\dem_data\cdnh44m_v3r1\uttarakhand_terrain_features.csv", index=False)
print("✓ Saved to CSV")


Uttarakhand catchments rebuilt: 119
✓ Saved to CSV


In [18]:
combined_terrain_features = pd.concat([catchment_features, assam_features], ignore_index=True)
combined_terrain_features['region'] = combined_terrain_features['catchment_id'].apply(
    lambda x: 'Uttarakhand' if x.startswith('C') else 'Assam'
)

print("Total combined catchments:", len(combined_terrain_features))
print(combined_terrain_features['region'].value_counts())

combined_terrain_features.to_csv(r"V:\flash_flood_ml\data\raw\terrain_features_combined.csv", index=False)
print("\n✓ Saved to terrain_features_combined.csv")
combined_terrain_features.head()

Total combined catchments: 1050
region
Assam          931
Uttarakhand    119
Name: count, dtype: int64

✓ Saved to terrain_features_combined.csv


,catchment_id,pixel_count,area_km2,mean_slope,mean_flow_accum,max_flow_accum,drainage_density,region
0,C1,1722,1.550,67.019997,47.029999,1722.0,0.213,Uttarakhand
1,C2,1524,1.372,72.199997,31.790001,1524.0,0.219,Uttarakhand
2,C3,2046,1.841,73.620003,37.750000,2046.0,0.407,Uttarakhand
3,C4,1230,1.107,70.610001,39.680000,1230.0,0.027,Uttarakhand
4,C5,2450,2.205,71.529999,38.959999,2450.0,0.259,Uttarakhand


In [1]:
import pandas as pd

uk_rain = pd.read_csv(r"V:\flash_flood_ml\data\raw\uttarakhand_rainfall_clean.csv")
assam_rain = pd.read_csv(r"V:\flash_flood_ml\data\raw\assam_rainfall_clean.csv")

for name, df in [("Uttarakhand", uk_rain), ("Assam", assam_rain)]:
    print(f"--- {name} ---")
    print(df.shape)
    print(df.dtypes)
    print(df.head(3))
    print()

FileNotFoundError: [Errno 2] No such file or directory: 'V:\\flash_flood_ml\\data\\raw\\uttarakhand_rainfall_clean.csv'

In [3]:
import os

for f in os.listdir("V:\\"):
    if "rainfall" in f.lower() or "rai" in f.lower():
        print(f)

assam_rainfall_clean.csv
uttarakhand_rainfall_clean.csv


In [4]:
import pandas as pd

uk_rain = pd.read_csv(r"V:\uttarakhand_rainfall_clean.csv")
assam_rain = pd.read_csv(r"V:\assam_rainfall_clean.csv")

for name, df in [("Uttarakhand", uk_rain), ("Assam", assam_rain)]:
    print(f"--- {name} ---")
    print(df.shape)
    print(df.dtypes)
    print(df.head(3))
    print()

--- Uttarakhand ---
(404006, 9)
date             object
lat             float64
lon             float64
rainfall_mm     float64
grid_cell_id     object
rainfall_1d     float64
rainfall_3d     float64
rainfall_7d     float64
rainfall_30d    float64
dtype: object
         date   lat   lon  rainfall_mm grid_cell_id  rainfall_1d  rainfall_3d  \
0  2019-01-01  28.5  77.5          0.0    28.5_77.5          0.0          0.0   
1  2019-01-02  28.5  77.5          0.0    28.5_77.5          0.0          0.0   
2  2019-01-03  28.5  77.5          0.0    28.5_77.5          0.0          0.0   

   rainfall_7d  rainfall_30d  
0          0.0           0.0  
1          0.0           0.0  
2          0.0           0.0  

--- Assam ---
(800341, 9)
date             object
lat             float64
lon             float64
rainfall_mm     float64
grid_cell_id     object
rainfall_1d     float64
rainfall_3d     float64
rainfall_7d     float64
rainfall_30d    float64
dtype: object
         date   lat    lon  rain

In [5]:
uk_rain['date'] = pd.to_datetime(uk_rain['date'])
assam_rain['date'] = pd.to_datetime(assam_rain['date'])

# Ek grid-cell pick karke rolling-calc verify karo
sample = uk_rain[uk_rain['grid_cell_id'] == uk_rain['grid_cell_id'].iloc[0]].sort_values('date').reset_index(drop=True)

print("Sample grid-cell verification:")
print(sample[['date', 'rainfall_mm', 'rainfall_1d', 'rainfall_3d', 'rainfall_7d']].head(10))

# Manually check karo: rainfall_3d ka 4th row = last 3 din ka sum hona chahiye
manual_3d = sample['rainfall_mm'].iloc[1:4].sum()
print("\nManual 3-day sum (rows 1-3):", manual_3d)
print("File mein rainfall_3d (row 3):", sample['rainfall_3d'].iloc[3])

Sample grid-cell verification:
        date  rainfall_mm  rainfall_1d  rainfall_3d  rainfall_7d
0 2019-01-01          0.0          0.0          0.0          0.0
1 2019-01-02          0.0          0.0          0.0          0.0
2 2019-01-03          0.0          0.0          0.0          0.0
3 2019-01-04          0.0          0.0          0.0          0.0
4 2019-01-05          0.0          0.0          0.0          0.0
5 2019-01-06          0.0          0.0          0.0          0.0
6 2019-01-07          0.0          0.0          0.0          0.0
7 2019-01-08          0.0          0.0          0.0          0.0
8 2019-01-09          0.0          0.0          0.0          0.0
9 2019-01-10          0.0          0.0          0.0          0.0

Manual 3-day sum (rows 1-3): 0.0
File mein rainfall_3d (row 3): 0.0


In [7]:
# Monsoon-season ka data dhoondo (July-August, jahan rainfall hone ki possibility zyada hai)
sample_monsoon = uk_rain[uk_rain['grid_cell_id'] == uk_rain['grid_cell_id'].iloc[0]].sort_values('date').reset_index(drop=True)
sample_monsoon = sample_monsoon[sample_monsoon['date'].dt.month.isin([7,8])]
sample_monsoon = sample_monsoon[sample_monsoon['rainfall_mm'] > 0].reset_index(drop=True)

print(sample_monsoon[['date', 'rainfall_mm', 'rainfall_1d', 'rainfall_3d', 'rainfall_7d']].head(10))

        date  rainfall_mm  rainfall_1d  rainfall_3d  rainfall_7d
0 2019-07-08     4.138261     4.138261     4.138261     4.138261
1 2019-07-09     8.461513     8.461513    12.599773    12.599773
2 2019-07-10     1.863573     1.863573    14.463346    14.463346
3 2019-07-16    25.245913    25.245913    25.245913    27.109485
4 2019-07-17    20.192215    20.192215    45.438128    45.438128
5 2019-07-18     1.978460     1.978460    47.416588    47.416588
6 2019-07-21     0.464740     0.464740     0.464740    47.881328
7 2019-07-22     2.558320     2.558320     3.023060    50.439648
8 2019-07-24     1.109940     1.109940     3.668260     6.111460
9 2019-07-25     2.235858     2.235858     3.345798     6.368858


In [9]:
import pandas as pd

# Terrain-features (already saved)
combined_terrain_features = pd.read_csv(r"V:\flash_flood_ml\data\raw\terrain_features_combined.csv")

# Rainfall + proxy files
uk_full = pd.read_csv(r"V:\uttarakhand_rainfall_with_proxies.csv")
assam_full = pd.read_csv(r"V:\assam_rainfall_with_proxies.csv")



print("Terrain shape:", combined_terrain_features.shape)
print("UK rainfall shape:", uk_full.shape)
print("Assam rainfall shape:", assam_full.shape)
print()
print(combined_terrain_features[['catchment_id','area_km2','region']].head())

Terrain shape: (1050, 8)
UK rainfall shape: (404006, 12)
Assam rainfall shape: (800341, 12)

  catchment_id  area_km2       region
0           C1     1.550  Uttarakhand
1           C2     1.372  Uttarakhand
2           C3     1.841  Uttarakhand
3           C4     1.107  Uttarakhand
4           C5     2.205  Uttarakhand


In [10]:
import rasterio
import numpy as np

def get_catchment_centers(catchment_tif_path, catchment_ids, prefix):
    with rasterio.open(catchment_tif_path) as ds:
        catchment_data = ds.read(1)
        transform = ds.transform
    
    records = []
    for cid_str in catchment_ids:
        cid = int(cid_str.replace(prefix, ""))
        mask = (catchment_data == cid)
        rows, cols = np.where(mask)
        if len(rows) == 0:
            continue
        center_row = rows.mean()
        center_col = cols.mean()
        # Pixel-coords ko geographic lat/lon mein convert karo
        lon, lat = rasterio.transform.xy(transform, center_row, center_col)
        records.append({'catchment_id': cid_str, 'center_lat': lat, 'center_lon': lon})
    
    return pd.DataFrame(records)

# Uttarakhand catchments ke centers
uk_ids = combined_terrain_features[combined_terrain_features['region']=='Uttarakhand']['catchment_id'].tolist()
uk_centers = get_catchment_centers(
    r"V:\flash_flood_ml\dem_data\cdnh44m_v3r1\catchments_v3.tif", uk_ids, "C"
)

# Assam catchments ke centers
assam_ids = combined_terrain_features[combined_terrain_features['region']=='Assam']['catchment_id'].tolist()
assam_centers = get_catchment_centers(
    r"V:\flash_flood_ml\data\raw\assam_catchments.tif", assam_ids, "A"
)

catchment_centers = pd.concat([uk_centers, assam_centers], ignore_index=True)
print("Centers computed:", len(catchment_centers))
catchment_centers.head()

Centers computed: 1050


,catchment_id,center_lat,center_lon
0,C1,29.996150,78.768457
1,C2,29.986513,78.535351
2,C3,29.987183,78.440383
3,C4,29.992208,78.923830
4,C5,29.989151,78.750153


In [11]:
combined_terrain_features = combined_terrain_features.merge(catchment_centers, on='catchment_id', how='left')
print("Missing centers:", combined_terrain_features['center_lat'].isnull().sum())
combined_terrain_features.head()

Missing centers: 0


,catchment_id,pixel_count,area_km2,mean_slope,mean_flow_accum,max_flow_accum,drainage_density,region,center_lat,center_lon
0,C1,1722,1.550,67.02,47.03,1722.0,0.213,Uttarakhand,29.996150,78.768457
1,C2,1524,1.372,72.20,31.79,1524.0,0.219,Uttarakhand,29.986513,78.535351
2,C3,2046,1.841,73.62,37.75,2046.0,0.407,Uttarakhand,29.987183,78.440383
3,C4,1230,1.107,70.61,39.68,1230.0,0.027,Uttarakhand,29.992208,78.923830
4,C5,2450,2.205,71.53,38.96,2450.0,0.259,Uttarakhand,29.989151,78.750153


In [12]:
from scipy.spatial import cKDTree

def find_nearest_grid(catchment_df, rainfall_df, region_name):
    region_catchments = catchment_df[catchment_df['region'] == region_name].copy()
    
    # Unique grid-cells nikalo (lat, lon pairs)
    unique_grids = rainfall_df[['grid_cell_id', 'lat', 'lon']].drop_duplicates().reset_index(drop=True)
    
    # KDTree banao fast nearest-neighbor search ke liye
    grid_coords = unique_grids[['lat', 'lon']].values
    tree = cKDTree(grid_coords)
    
    catchment_coords = region_catchments[['center_lat', 'center_lon']].values
    distances, indices = tree.query(catchment_coords)
    
    region_catchments['nearest_grid_cell_id'] = unique_grids.iloc[indices]['grid_cell_id'].values
    region_catchments['grid_distance_deg'] = distances
    
    return region_catchments

uk_matched = find_nearest_grid(combined_terrain_features, uk_full, 'Uttarakhand')
assam_matched = find_nearest_grid(combined_terrain_features, assam_full, 'Assam')

catchment_grid_map = pd.concat([uk_matched, assam_matched], ignore_index=True)

print("Matched catchments:", len(catchment_grid_map))
print("\nDistance stats (degrees, ~1° ≈ 111km):")
print(catchment_grid_map['grid_distance_deg'].describe())
catchment_grid_map[['catchment_id','center_lat','center_lon','nearest_grid_cell_id','grid_distance_deg']].head()

Matched catchments: 1050

Distance stats (degrees, ~1° ≈ 111km):
count    1050.000000
mean        0.194108
std         0.195720
min         0.002481
25%         0.080843
50%         0.120079
75%         0.219125
max         1.001656
Name: grid_distance_deg, dtype: float64


,catchment_id,center_lat,center_lon,nearest_grid_cell_id,grid_distance_deg
0,C1,29.996150,78.768457,30.0_78.75,0.018854
1,C2,29.986513,78.535351,30.0_78.5,0.037836
2,C3,29.987183,78.440383,30.0_78.5,0.060979
3,C4,29.992208,78.923830,30.0_79.0,0.076567
4,C5,29.989151,78.750153,30.0_78.75,0.010850


In [13]:
outliers = catchment_grid_map[catchment_grid_map['grid_distance_deg'] > 0.3].sort_values('grid_distance_deg', ascending=False)
print("Catchments with large grid-distance (>0.3°, ~33km):")
print(outliers[['catchment_id','region','center_lat','center_lon','nearest_grid_cell_id','grid_distance_deg']].head(15))
print(f"\nTotal outliers: {len(outliers)} out of {len(catchment_grid_map)}")

Catchments with large grid-distance (>0.3°, ~33km):
    catchment_id region  center_lat  center_lon nearest_grid_cell_id  \
802         A703  Assam   24.227098   90.248606          24.25_91.25   
624         A507  Assam   24.242201   90.249705          24.25_91.25   
500         A382  Assam   24.273154   90.158235          25.25_90.25   
457         A339  Assam   24.288055   90.258174          25.25_90.25   
458         A340  Assam   24.400233   90.024963          25.25_90.25   
818         A719  Assam   24.406099   90.151871          25.25_90.25   
814         A715  Assam   24.410891   90.390134           25.25_90.5   
623         A506  Assam   24.005694   90.404722           24.0_91.25   
534         A416  Assam   24.406170   90.420433          24.25_91.25   
526         A408  Assam   24.440996   90.015946          25.25_90.25   
637         A522  Assam   24.425939   90.277431          25.25_90.25   
731         A620  Assam   24.446499   90.109704          25.25_90.25   
400         

In [2]:
# Assam rainfall grid ke actual unique lat/lon points dekho
import pandas as pd

assam_full = pd.read_csv(r"V:\assam_rainfall_with_proxies.csv")
assam_grid_points = assam_full[['lat','lon']].drop_duplicates().sort_values(['lat','lon'])
print("Total unique grid points:", len(assam_grid_points))
print("\nLat values available:", sorted(assam_grid_points['lat'].unique()))
print("\nLon values available:", sorted(assam_grid_points['lon'].unique()))


Total unique grid points: 313

Lat values available: [np.float64(24.0), np.float64(24.25), np.float64(24.5), np.float64(24.75), np.float64(25.0), np.float64(25.25), np.float64(25.5), np.float64(25.75), np.float64(26.0), np.float64(26.25), np.float64(26.5), np.float64(26.75), np.float64(27.0), np.float64(27.25), np.float64(27.5), np.float64(27.75), np.float64(28.0)]

Lon values available: [np.float64(89.5), np.float64(89.75), np.float64(90.0), np.float64(90.25), np.float64(90.5), np.float64(90.75), np.float64(91.0), np.float64(91.25), np.float64(91.5), np.float64(91.75), np.float64(92.0), np.float64(92.25), np.float64(92.5), np.float64(92.75), np.float64(93.0), np.float64(93.25), np.float64(93.5), np.float64(93.75), np.float64(94.0), np.float64(94.25), np.float64(94.5), np.float64(94.75), np.float64(95.0), np.float64(95.25), np.float64(95.5), np.float64(95.75), np.float64(96.0), np.float64(96.25), np.float64(96.5)]


In [3]:
# Check karo kya specific corner-combinations exist karte hain
corner_check = assam_full[
    (assam_full['lat'].between(24.0, 24.75)) & 
    (assam_full['lon'].between(90.0, 90.75))
][['lat','lon','grid_cell_id']].drop_duplicates()

print("Grid-cells available in 24.0-24.75°N, 90.0-90.75°E corner:")
print(corner_check)

Grid-cells available in 24.0-24.75°N, 90.0-90.75°E corner:
Empty DataFrame
Columns: [lat, lon, grid_cell_id]
Index: []


In [5]:
import pandas as pd
import rasterio
import numpy as np
from scipy.spatial import cKDTree

# ===== 1. Sab base-data reload karo =====
combined_terrain_features = pd.read_csv(r"V:\flash_flood_ml\data\raw\terrain_features_combined.csv")
uk_full = pd.read_csv(r"V:\uttarakhand_rainfall_with_proxies.csv")
assam_full = pd.read_csv(r"V:\assam_rainfall_with_proxies.csv")

# ===== 2. Catchment-centers nikalo =====
def get_catchment_centers(catchment_tif_path, catchment_ids, prefix):
    with rasterio.open(catchment_tif_path) as ds:
        catchment_data = ds.read(1)
        transform = ds.transform
    records = []
    for cid_str in catchment_ids:
        cid = int(cid_str.replace(prefix, ""))
        mask = (catchment_data == cid)
        rows, cols = np.where(mask)
        if len(rows) == 0:
            continue
        center_row, center_col = rows.mean(), cols.mean()
        lon, lat = rasterio.transform.xy(transform, center_row, center_col)
        records.append({'catchment_id': cid_str, 'center_lat': lat, 'center_lon': lon})
    return pd.DataFrame(records)

uk_ids = combined_terrain_features[combined_terrain_features['region']=='Uttarakhand']['catchment_id'].tolist()
uk_centers = get_catchment_centers(r"V:\flash_flood_ml\dem_data\cdnh44m_v3r1\catchments_v3.tif", uk_ids, "C")

assam_ids = combined_terrain_features[combined_terrain_features['region']=='Assam']['catchment_id'].tolist()
assam_centers = get_catchment_centers(r"V:\flash_flood_ml\data\raw\assam_catchments.tif", assam_ids, "A")

catchment_centers = pd.concat([uk_centers, assam_centers], ignore_index=True)
combined_terrain_features = combined_terrain_features.merge(catchment_centers, on='catchment_id', how='left')

# ===== 3. Nearest rainfall grid-cell match karo =====
def find_nearest_grid(catchment_df, rainfall_df, region_name):
    region_catchments = catchment_df[catchment_df['region'] == region_name].copy()
    unique_grids = rainfall_df[['grid_cell_id', 'lat', 'lon']].drop_duplicates().reset_index(drop=True)
    tree = cKDTree(unique_grids[['lat', 'lon']].values)
    distances, indices = tree.query(region_catchments[['center_lat', 'center_lon']].values)
    region_catchments['nearest_grid_cell_id'] = unique_grids.iloc[indices]['grid_cell_id'].values
    region_catchments['grid_distance_deg'] = distances
    return region_catchments

uk_matched = find_nearest_grid(combined_terrain_features, uk_full, 'Uttarakhand')
assam_matched = find_nearest_grid(combined_terrain_features, assam_full, 'Assam')
catchment_grid_map = pd.concat([uk_matched, assam_matched], ignore_index=True)

# ===== 4. Match-quality flag karo =====
DISTANCE_THRESHOLD = 0.3
catchment_grid_map['rainfall_match_quality'] = catchment_grid_map['grid_distance_deg'].apply(
    lambda d: 'good' if d <= DISTANCE_THRESHOLD else 'low_confidence'
)

print(catchment_grid_map['rainfall_match_quality'].value_counts())

# ===== 5. Turant save kar do taaki dobara na banana pade =====
catchment_grid_map.to_csv(r"V:\flash_flood_ml\data\raw\catchment_grid_map.csv", index=False)
print("\n✓ Saved catchment_grid_map.csv")

rainfall_match_quality
good              867
low_confidence    183
Name: count, dtype: int64

✓ Saved catchment_grid_map.csv


In [6]:
# Sirf good-quality catchments rakho
good_catchments = catchment_grid_map[catchment_grid_map['rainfall_match_quality'] == 'good'].copy()
print("Usable catchments:", len(good_catchments))

# Har catchment ko uske rainfall-timeseries se merge karo (grid_cell_id se)
uk_rainfall_subset = uk_full.rename(columns={'grid_cell_id': 'nearest_grid_cell_id'})
assam_rainfall_subset = assam_full.rename(columns={'grid_cell_id': 'nearest_grid_cell_id'})
all_rainfall = pd.concat([uk_rainfall_subset, assam_rainfall_subset], ignore_index=True)

# Static terrain-info (per catchment, ek row) ko catchment-metadata se alag rakho
catchment_static = good_catchments[[
    'catchment_id', 'region', 'area_km2', 'mean_slope', 'mean_flow_accum', 
    'max_flow_accum', 'drainage_density', 'nearest_grid_cell_id'
]]

# Merge karo — har catchment ke saath uski poori rainfall-timeseries jud jayegi
final_feature_table = catchment_static.merge(
    all_rainfall[['date', 'nearest_grid_cell_id', 'rainfall_mm', 'rainfall_1d', 
                  'rainfall_3d', 'rainfall_7d', 'rainfall_30d', 
                  'soil_saturation_proxy', 'ndvi']],
    on='nearest_grid_cell_id',
    how='left'
)

print("\nFinal feature-table shape:", final_feature_table.shape)
print("Unique catchments:", final_feature_table['catchment_id'].nunique())
print("Date range:", final_feature_table['date'].min(), "to", final_feature_table['date'].max())
final_feature_table.head()

Usable catchments: 867

Final feature-table shape: (2216919, 16)
Unique catchments: 867
Date range: 2019-01-01 to 2025-12-31


,catchment_id,region,area_km2,mean_slope,mean_flow_accum,max_flow_accum,drainage_density,nearest_grid_cell_id,date,rainfall_mm,rainfall_1d,rainfall_3d,rainfall_7d,rainfall_30d,soil_saturation_proxy,ndvi
0,C1,Uttarakhand,1.55,67.02,47.03,1722.0,0.213,30.0_78.75,2019-01-01,0.0,0.0,0.0,0.0,0.0,0.0,0.65
1,C1,Uttarakhand,1.55,67.02,47.03,1722.0,0.213,30.0_78.75,2019-01-02,0.0,0.0,0.0,0.0,0.0,0.0,0.65
2,C1,Uttarakhand,1.55,67.02,47.03,1722.0,0.213,30.0_78.75,2019-01-03,0.0,0.0,0.0,0.0,0.0,0.0,0.65
3,C1,Uttarakhand,1.55,67.02,47.03,1722.0,0.213,30.0_78.75,2019-01-04,0.0,0.0,0.0,0.0,0.0,0.0,0.65
4,C1,Uttarakhand,1.55,67.02,47.03,1722.0,0.213,30.0_78.75,2019-01-05,0.0,0.0,0.0,0.0,0.0,0.0,0.65


In [7]:
final_feature_table.to_csv(r"V:\flash_flood_ml\data\raw\final_feature_table_no_target.csv", index=False)
print("✓ Saved final_feature_table_no_target.csv")
print("File size approx:", final_feature_table.memory_usage(deep=True).sum() / 1e6, "MB in memory")

✓ Saved final_feature_table_no_target.csv
File size approx: 783.705286 MB in memory


In [8]:
# Dharali event ke coordinates se nearest catchment dhoondo
dharali_lat, dharali_lon = 31.0408, 78.7811
event_date = '2025-08-05'

good_catchments['dist_to_dharali'] = np.sqrt(
    (good_catchments['center_lat'] - dharali_lat)**2 + 
    (good_catchments['center_lon'] - dharali_lon)**2
)

nearest_to_dharali = good_catchments.nsmallest(5, 'dist_to_dharali')
print(nearest_to_dharali[['catchment_id','center_lat','center_lon','dist_to_dharali']])

  catchment_id  center_lat  center_lon  dist_to_dharali
0           C1   29.996150   78.768457         1.044726
4           C5   29.989151   78.750153         1.052104
8           C9   29.982812   78.761803         1.058164
3           C4   29.992208   78.923830         1.058262
6           C7   29.980550   78.826767         1.061233


In [9]:
print("DEM tile coverage: Lat 29.0-30.0, Lon 78.0-79.0")
print("Dharali actual location: Lat 31.04, Lon 78.78")
print("Verdict: Dharali OUTSIDE current DEM tile — need adjacent tile to the north (79E30N)")

DEM tile coverage: Lat 29.0-30.0, Lon 78.0-79.0
Dharali actual location: Lat 31.04, Lon 78.78
Verdict: Dharali OUTSIDE current DEM tile — need adjacent tile to the north (79E30N)


In [10]:
import pandas as pd

uk_real = pd.read_csv(r"V:\uttarakhand_full_features_REAL.csv")  # apna exact path check kar lo
uk_real['date'] = pd.to_datetime(uk_real['date'])

print(uk_real.shape)
print(uk_real['terrain_source'].value_counts())
uk_real.head()

(404006, 15)
terrain_source
real_dem_cartodem_v3                      296612
regional_estimate_outside_dem_coverage    107394
Name: count, dtype: int64


,date,lat,lon,rainfall_mm,grid_cell_id,rainfall_1d,rainfall_3d,rainfall_7d,rainfall_30d,soil_saturation_proxy,ndvi,ndvi_source,slope_mean,flow_accumulation,terrain_source
0,2019-01-01,28.5,77.5,0.0,28.5_77.5,0.0,0.0,0.0,0.0,0.0,0.65,placeholder_static,27.490802,3805.111137,regional_estimate_outside_dem_coverage
1,2019-01-02,28.5,77.5,0.0,28.5_77.5,0.0,0.0,0.0,0.0,0.0,0.65,placeholder_static,39.014286,1672.346518,regional_estimate_outside_dem_coverage
2,2019-01-03,28.5,77.5,0.0,28.5_77.5,0.0,0.0,0.0,0.0,0.0,0.65,placeholder_static,34.639879,2106.579178,regional_estimate_outside_dem_coverage
3,2019-01-04,28.5,77.5,0.0,28.5_77.5,0.0,0.0,0.0,0.0,0.0,0.65,placeholder_static,31.973170,3196.388448,regional_estimate_outside_dem_coverage
4,2019-01-05,28.5,77.5,0.0,28.5_77.5,0.0,0.0,0.0,0.0,0.0,0.65,placeholder_static,23.120373,2799.861369,regional_estimate_outside_dem_coverage


In [11]:
# Target column banate hain
uk_real['flood_event'] = 0

# Dharali event window — 4-5 August ko positive mark karo (event + immediate lead-up)
event_grid = '31.0_78.75'
event_window = pd.date_range('2025-08-04', '2025-08-05')

mask = (uk_real['grid_cell_id'] == event_grid) & (uk_real['date'].isin(event_window))
uk_real.loc[mask, 'flood_event'] = 1

print("Total positive labels:", uk_real['flood_event'].sum())
print("\nPositive rows:")
print(uk_real[uk_real['flood_event']==1][['date','grid_cell_id','rainfall_mm','rainfall_3d','rainfall_7d']])

Total positive labels: 2

Positive rows:
             date grid_cell_id  rainfall_mm  rainfall_3d  rainfall_7d
350159 2025-08-04   31.0_78.75    27.948360    33.192000    85.665090
350160 2025-08-05   31.0_78.75    10.827349    41.231561    94.837106


In [2]:
import pandas as pd
assam_real = pd.read_csv(r"V:\assam_full_features_REAL.csv")  # apna exact path confirm kar lo
assam_real['date'] = pd.to_datetime(assam_real['date'])

print("Assam shape:", assam_real.shape)
print(assam_real['terrain_source'].value_counts())

Assam shape: (800341, 15)
terrain_source
real_dem_cartodem_v3                      677605
regional_estimate_outside_dem_coverage    122736
Name: count, dtype: int64


In [3]:
approx_events = [
    {'name': 'Cachar 2023', 'date': '2023-06-14', 'lat': 24.83, 'lon': 92.78},
    {'name': 'Nagaon 2024', 'date': '2024-05-31', 'lat': 26.35, 'lon': 92.68},
    {'name': 'Lakhimpur 2023', 'date': '2023-06-14', 'lat': 27.24, 'lon': 94.11},
]

assam_real['flood_event'] = 0

grids = assam_real[['lat','lon','grid_cell_id']].drop_duplicates()

for ev in approx_events:
    grids_copy = grids.copy()
    grids_copy['dist'] = ((grids_copy.lat-ev['lat'])**2 + (grids_copy.lon-ev['lon'])**2)**0.5
    nearest = grids_copy.nsmallest(1, 'dist').iloc[0]
    print(f"{ev['name']}: nearest grid={nearest['grid_cell_id']}, distance={nearest['dist']:.3f}°")
    
    event_window = pd.date_range(
        pd.to_datetime(ev['date']) - pd.Timedelta(days=1),
        pd.to_datetime(ev['date'])
    )
    mask = (assam_real['grid_cell_id'] == nearest['grid_cell_id']) & (assam_real['date'].isin(event_window))
    assam_real.loc[mask, 'flood_event'] = 1

print("\nTotal Assam positive labels:", assam_real['flood_event'].sum())
print(assam_real[assam_real['flood_event']==1][['date','grid_cell_id','rainfall_mm','rainfall_3d','rainfall_7d']])

Cachar 2023: nearest grid=24.75_92.75, distance=0.085°
Nagaon 2024: nearest grid=26.25_92.75, distance=0.122°
Lakhimpur 2023: nearest grid=27.25_94.0, distance=0.110°

Total Assam positive labels: 6
             date grid_cell_id  rainfall_mm  rainfall_3d  rainfall_7d
98790  2023-06-13  24.75_92.75    31.108166    62.626675    68.606134
98791  2023-06-14  24.75_92.75     1.007279    36.619834    69.613414
390640 2024-05-30  26.25_92.75    25.699394    99.491360   120.523944
390641 2024-05-31  26.25_92.75    16.801262    82.388401   131.239780
620418 2023-06-13   27.25_94.0    28.306358    91.416375   109.979723
620419 2023-06-14   27.25_94.0    49.407482   128.679855   159.387205


In [4]:
uk_real['region'] = 'Uttarakhand'
assam_real['region'] = 'Assam'

combined_real = pd.concat([uk_real, assam_real], ignore_index=True)

print("Combined shape:", combined_real.shape)
print("Total positive labels:", combined_real['flood_event'].sum())
print("Positive rate:", round(combined_real['flood_event'].mean()*100, 6), "%")
print(combined_real['region'].value_counts())

# Save karo taaki dobara na banana pade
combined_real.to_csv(r"V:\flash_flood_ml\data\raw\combined_real_features_with_target.csv", index=False)
print("\n✓ Saved combined_real_features_with_target.csv")

NameError: name 'uk_real' is not defined

In [5]:
import pandas as pd

# ===== Uttarakhand reload + target-label =====
uk_real = pd.read_csv(r"V:\uttarakhand_full_features_REAL.csv")
uk_real['date'] = pd.to_datetime(uk_real['date'])

uk_real['flood_event'] = 0
event_grid = '31.0_78.75'
event_window = pd.date_range('2025-08-04', '2025-08-05')
mask = (uk_real['grid_cell_id'] == event_grid) & (uk_real['date'].isin(event_window))
uk_real.loc[mask, 'flood_event'] = 1

# ===== Assam reload + target-label =====
assam_real = pd.read_csv(r"V:\assam_full_features_REAL.csv")
assam_real['date'] = pd.to_datetime(assam_real['date'])

assam_real['flood_event'] = 0
approx_events = [
    {'name': 'Cachar 2023', 'date': '2023-06-14', 'lat': 24.83, 'lon': 92.78},
    {'name': 'Nagaon 2024', 'date': '2024-05-31', 'lat': 26.35, 'lon': 92.68},
    {'name': 'Lakhimpur 2023', 'date': '2023-06-14', 'lat': 27.24, 'lon': 94.11},
]
grids = assam_real[['lat','lon','grid_cell_id']].drop_duplicates()

for ev in approx_events:
    grids_copy = grids.copy()
    grids_copy['dist'] = ((grids_copy.lat-ev['lat'])**2 + (grids_copy.lon-ev['lon'])**2)**0.5
    nearest = grids_copy.nsmallest(1, 'dist').iloc[0]
    event_window = pd.date_range(
        pd.to_datetime(ev['date']) - pd.Timedelta(days=1),
        pd.to_datetime(ev['date'])
    )
    mask = (assam_real['grid_cell_id'] == nearest['grid_cell_id']) & (assam_real['date'].isin(event_window))
    assam_real.loc[mask, 'flood_event'] = 1

# ===== Combine =====
uk_real['region'] = 'Uttarakhand'
assam_real['region'] = 'Assam'
combined_real = pd.concat([uk_real, assam_real], ignore_index=True)

print("Combined shape:", combined_real.shape)
print("Total positive labels:", combined_real['flood_event'].sum())
print(combined_real['region'].value_counts())

combined_real.to_csv(r"V:\flash_flood_ml\data\raw\combined_real_features_with_target.csv", index=False)
print("\n✓ Saved combined_real_features_with_target.csv")

Combined shape: (1204347, 17)
Total positive labels: 8
region
Assam          800341
Uttarakhand    404006
Name: count, dtype: int64

✓ Saved combined_real_features_with_target.csv


In [6]:
new_events = [
    # Uttarakhand
    {'name': 'Joshimath cloudburst 2023', 'date': '2023-06-24', 'lat': 30.55, 'lon': 79.57, 'region': 'Uttarakhand'},
    {'name': 'Almora flashflood 2023', 'date': '2023-09-15', 'lat': 29.70, 'lon': 79.50, 'region': 'Uttarakhand'},
    {'name': 'Chamoli/Rishiganga 2021', 'date': '2021-02-07', 'lat': 30.56, 'lon': 79.73, 'region': 'Uttarakhand'},
    {'name': 'Dehradun cloudburst 2022', 'date': '2022-08-20', 'lat': 30.35, 'lon': 78.08, 'region': 'Uttarakhand'},
    # Assam
    {'name': 'Silchar flood 2022', 'date': '2022-06-19', 'lat': 24.83, 'lon': 92.78, 'region': 'Assam'},
    {'name': 'Cachar/Dhubri flood 2022', 'date': '2022-05-23', 'lat': 24.83, 'lon': 92.78, 'region': 'Assam'},
    {'name': 'Assam floods 2024', 'date': '2024-05-26', 'lat': 26.0, 'lon': 92.5, 'region': 'Assam'},
    {'name': 'Assam floods 2020 peak', 'date': '2020-07-21', 'lat': 26.0, 'lon': 92.5, 'region': 'Assam'},
]

added_count = 0
for ev in new_events:
    region_df = combined_real[combined_real['region'] == ev['region']]
    grids = region_df[['lat','lon','grid_cell_id']].drop_duplicates().copy()
    grids['dist'] = ((grids.lat-ev['lat'])**2 + (grids.lon-ev['lon'])**2)**0.5
    nearest = grids.nsmallest(1, 'dist').iloc[0]
    
    if nearest['dist'] > 0.5:  # bahut door hai to skip karo
        print(f"SKIP {ev['name']}: nearest grid too far ({nearest['dist']:.2f}°)")
        continue
    
    event_window = pd.date_range(
        pd.to_datetime(ev['date']) - pd.Timedelta(days=1),
        pd.to_datetime(ev['date'])
    )
    mask = (combined_real['grid_cell_id'] == nearest['grid_cell_id']) & (combined_real['date'].isin(event_window)) & (combined_real['region'] == ev['region'])
    combined_real.loc[mask, 'flood_event'] = 1
    added_count += mask.sum()
    print(f"✓ {ev['name']}: grid={nearest['grid_cell_id']}, dist={nearest['dist']:.3f}°, rows-marked={mask.sum()}")

print(f"\nTotal new positive-rows added: {added_count}")
print(f"Grand total positive labels now: {combined_real['flood_event'].sum()}")

combined_real.to_csv(r"V:\flash_flood_ml\data\raw\combined_real_features_with_target.csv", index=False)
print("✓ Re-saved with updated targets")

✓ Joshimath cloudburst 2023: grid=30.5_79.5, dist=0.086°, rows-marked=2
✓ Almora flashflood 2023: grid=29.75_79.5, dist=0.050°, rows-marked=2
✓ Chamoli/Rishiganga 2021: grid=30.5_79.75, dist=0.063°, rows-marked=2
✓ Dehradun cloudburst 2022: grid=30.25_78.0, dist=0.128°, rows-marked=2
✓ Silchar flood 2022: grid=24.75_92.75, dist=0.085°, rows-marked=2
✓ Cachar/Dhubri flood 2022: grid=24.75_92.75, dist=0.085°, rows-marked=2
✓ Assam floods 2024: grid=26.0_92.5, dist=0.000°, rows-marked=2
✓ Assam floods 2020 peak: grid=26.0_92.5, dist=0.000°, rows-marked=2

Total new positive-rows added: 16
Grand total positive labels now: 24
✓ Re-saved with updated targets


In [7]:
from sklearn.model_selection import train_test_split
import xgboost
from sklearn.metrics import classification_report, average_precision_score

feature_cols = ['rainfall_mm','rainfall_1d','rainfall_3d','rainfall_7d','rainfall_30d',
                 'soil_saturation_proxy','ndvi','slope_mean','flow_accumulation']

X = combined_real[feature_cols]
y = combined_real['flood_event']

# Stratified-split zaroori hai taaki positive-examples dono sides mein proportionally jaayein
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

print("Train positives:", y_train.sum(), "| Test positives:", y_test.sum())

model_real = xgboost.XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),
    eval_metric='aucpr', random_state=42
)
model_real.fit(X_train, y_train)

y_pred = model_real.predict(X_test)
y_proba = model_real.predict_proba(X_test)[:,1]

print("\n--- Real-Data Model Results ---")
print(classification_report(y_test, y_pred))
print("PR-AUC:", round(average_precision_score(y_test, y_proba), 4))

Train positives: 17 | Test positives: 7

--- Real-Data Model Results ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    361298
           1       0.00      0.00      0.00         7

    accuracy                           1.00    361305
   macro avg       0.50      0.50      0.50    361305
weighted avg       1.00      1.00      1.00    361305

PR-AUC: 0.0019


In [8]:
combined_real['pred_proba'] = model_real.predict_proba(X)[:,1]

test_positives_proba = combined_real.loc[X_test.index][combined_real.loc[X_test.index]['flood_event']==1]['pred_proba']
random_sample_proba = combined_real.loc[X_test.index][combined_real.loc[X_test.index]['flood_event']==0].sample(1000, random_state=1)['pred_proba']

print("Test-positives probability:", test_positives_proba.values)
print("\nRandom-negative-sample probability stats:")
print(random_sample_proba.describe())

Test-positives probability: [5.1575575e-05 1.3509501e-02 2.4354275e-05 1.3828827e-01 4.2398939e-01
 2.4354275e-05 5.2572051e-03]

Random-negative-sample probability stats:
count    1.000000e+03
mean     1.082038e-03
std      7.449693e-03
min      7.003745e-07
25%      6.738020e-06
50%      2.279993e-05
75%      1.042550e-04
max      1.165772e-01
Name: pred_proba, dtype: float64


In [3]:
import pandas as pd


# ===== 1. Sab base-data reload karo =====
uk_real = pd.read_csv(r"V:\uttarakhand_full_features_REAL.csv")
uk_real['date'] = pd.to_datetime(uk_real['date'])

assam_real = pd.read_csv(r"V:\assam_full_features_REAL.csv")
assam_real['date'] = pd.to_datetime(assam_real['date'])

# ===== 2. Already-saved combined-target-file load karo (agar exist karta hai) =====
combined_real = pd.read_csv(r"V:\flash_flood_ml\data\raw\combined_real_features_with_target.csv")
combined_real['date'] = pd.to_datetime(combined_real['date'])

print("Loaded combined_real:", combined_real.shape)
print("Current positive labels:", combined_real['flood_event'].sum())
# Chamoli/Rishiganga 2021 event ko unlabel karo — ye avalanche-triggered tha, rainfall-triggered nahi
event_grid = '30.5_79.75'
event_date = pd.to_datetime('2021-02-07')
event_window = pd.date_range(event_date - pd.Timedelta(days=1), event_date)

mask = (combined_real['grid_cell_id'] == event_grid) & (combined_real['date'].isin(event_window)) & (combined_real['region'] == 'Uttarakhand')
combined_real.loc[mask, 'flood_event'] = 0

print("Rows un-labeled:", mask.sum())
print("New total positive labels:", combined_real['flood_event'].sum())

# Re-save
combined_real.to_csv(r"V:\flash_flood_ml\data\raw\combined_real_features_with_target.csv", index=False)
print("✓ Re-saved with corrected targets")

Loaded combined_real: (1204347, 17)
Current positive labels: 24
Rows un-labeled: 2
New total positive labels: 22
✓ Re-saved with corrected targets


In [4]:
from sklearn.model_selection import train_test_split
import xgboost
from sklearn.metrics import classification_report, average_precision_score

feature_cols = ['rainfall_mm','rainfall_1d','rainfall_3d','rainfall_7d','rainfall_30d',
                 'soil_saturation_proxy','ndvi','slope_mean','flow_accumulation']

X = combined_real[feature_cols]
y = combined_real['flood_event']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

print("Train positives:", y_train.sum(), "| Test positives:", y_test.sum())

model_real = xgboost.XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),
    eval_metric='aucpr', random_state=42
)
model_real.fit(X_train, y_train)

y_pred = model_real.predict(X_test)
y_proba = model_real.predict_proba(X_test)[:,1]

print("\n--- Updated Real-Data Model Results ---")
print(classification_report(y_test, y_pred))
print("PR-AUC:", round(average_precision_score(y_test, y_proba), 4))

# Probability-ranking check (jaisa pehle kiya tha)
combined_real['pred_proba'] = model_real.predict_proba(X)[:,1]
test_positives_proba = combined_real.loc[X_test.index][combined_real.loc[X_test.index]['flood_event']==1]['pred_proba']
random_sample_proba = combined_real.loc[X_test.index][combined_real.loc[X_test.index]['flood_event']==0].sample(1000, random_state=1)['pred_proba']

print("\nTest-positives probability:", sorted(test_positives_proba.values, reverse=True))
print("Random-negative mean:", round(random_sample_proba.mean(), 6), "| max:", round(random_sample_proba.max(), 4))

Train positives: 15 | Test positives: 7

--- Updated Real-Data Model Results ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    361298
           1       0.01      0.14      0.03         7

    accuracy                           1.00    361305
   macro avg       0.51      0.57      0.51    361305
weighted avg       1.00      1.00      1.00    361305

PR-AUC: 0.0092

Test-positives probability: [np.float32(0.7881914), np.float32(0.13850503), np.float32(0.005713622), np.float32(0.0032373585), np.float32(0.00054040266), np.float32(0.0003052517), np.float32(3.0287822e-05)]
Random-negative mean: 0.000564 | max: 0.0924


In [5]:
final_training_data = combined_real.copy()
final_training_data.to_csv(r"V:\flash_flood_ml\data\raw\final_training_data.csv", index=False)
print("✓ Final training data saved:", final_training_data.shape)
print("Positive labels:", final_training_data['flood_event'].sum())


✓ Final training data saved: (1204347, 18)
Positive labels: 22
